# WCG 模型对Titanic乘客生存状况预测

WCG模型（Woman-Child-Group）

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [2]:
# 读取数据并合并
train = pd.read_csv('../titanic/train.csv')
test = pd.read_csv('../titanic/test.csv')

test['Survived'] = np.nan

allData = pd.concat([train, test], ignore_index=True)

In [3]:
allData['Title'] = 'man'
allData.loc[allData['Name'].str.contains('Master', na=False), 'Title'] = 'boy'
allData.loc[allData['Sex']=='female', 'Title'] = 'woman'

In [4]:
allData['Surname'] = allData['Name'].str.split(',').str[0].str.strip()

allData.loc[allData['Title']=='man', 'Surname'] = 'noGroup'

allData['SurnameFreq'] = allData.groupby('Surname')['Surname'].transform('count')

# allData.loc[allData['Title']=='man', 'Surname'] = 'noGroup'

# 计算每个姓氏的频率
# allData['SurnameFreq'] = allData.groupby('Surname')['Surname'].transform('count')

# 如果姓氏只出现一次，归为noGroup
allData.loc[allData['SurnameFreq']<=1,'Surname'] = 'noGroup'

In [5]:
allData.loc[allData.index[(allData['Title'] != "man") & (allData['SurnameFreq'] == 1)],:]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Surname,SurnameFreq
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,woman,noGroup,1
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,woman,noGroup,1
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,woman,noGroup,1
9,10,1.0,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,woman,noGroup,1
14,15,0.0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,NaN,S,woman,noGroup,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1288,1289,NaN,1,"Frolicher-Stehli, Mrs. Maxmillian (Margaretha ...",female,48.0,1,1,13567,79.2000,B41,C,woman,noGroup,1
1299,1300,NaN,3,"Riordan, Miss. Johanna Hannah""""",female,NaN,0,0,334915,7.7208,NaN,Q,woman,noGroup,1
1301,1302,NaN,3,"Naughton, Miss. Hannah",female,NaN,0,0,365237,7.7500,NaN,Q,woman,noGroup,1
1303,1304,NaN,3,"Henriksson, Miss. Jenny Lovisa",female,28.0,0,0,347086,7.7750,NaN,S,woman,noGroup,1


In [6]:
# # 找出所有非男士且分组为noGroup的乘客，填补缺失信息
for i in allData.index[(allData['Title'] != "man") & (allData['SurnameFreq'] == 1)]:
    # 找到和这些乘客相同Ticket编号的乘客，标为对应乘客的Surname
    same_ticket_surname = allData.loc[allData['Ticket'] == allData.loc[i, 'Ticket'],"Surname"]
    if not same_ticket_surname.empty:
        allData.loc[i, 'Surname'] = same_ticket_surname.iloc[0]

allData['Surname'] = allData['Surname'].fillna('noGroup')

In [7]:
# 计算 woman-child-group 生存率
allData['SurnameSurvival'] = np.nan

In [8]:
# 对训练集计算每个姓氏的生存率
train_survival = allData.iloc[:891].groupby("Surname")['Survived'].transform('mean')

allData.loc[:890, 'SurnameSurvival'] = train_survival

In [9]:
# 对test部分的数据用train中相同姓氏的survival rate 填充
for i in range(891, len(allData)):
    surname = allData.loc[i, 'Surname']
    survival_rate = allData.loc[(allData.index<891)&(allData['Surname']==surname), "SurnameSurvival"]
    if not survival_rate.empty:
        allData.loc[i, 'SurnameSurvival'] = survival_rate.iloc[0]

In [10]:
allData['predict'] = 0
allData.loc[allData['Title']=='woman','predict'] = 1
allData.loc[(allData['Title']=='boy')&(allData['SurnameSurvival']==1),'predict'] = 1
allData.loc[(allData['Title']=='woman')&(allData['SurnameSurvival']==0),'predict'] = 0

In [11]:
test_part = allData.iloc[891:]

In [12]:
y_test = test_part['predict']

In [13]:
truth = pd.read_csv('../titanic/truth.csv')
y_true = truth['Survived']

In [14]:
from sklearn.metrics import accuracy_score
accuracy_score(y_true, y_test)

0.8038277511961722

找到所有Female和boy的数据，比较Female和boy的预测率

In [15]:
truth

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [16]:
test_part_index = test_part[(test_part['Title']=='woman')&(test_part['Surname']=='noGroup')]['PassengerId']
y_true_female = truth.loc[truth['PassengerId'].isin(test_part_index), :]

In [17]:
y_test_female = test_part.loc[test_part['PassengerId'].isin(test_part_index), ['PassengerId', 'predict']]
y_test_female

,PassengerId,predict
892,893,1
899,900,1
903,904,1
905,906,1
906,907,1
...,...,...
1288,1289,1
1299,1300,1
1301,1302,1
1303,1304,1


In [24]:
y_test.reset_index(drop=True, inplace=True)
y_test = pd.DataFrame({'Survived':y_test})
y_test['PassengerId'] = truth['PassengerId']
y_test.to_csv('titanic/wcg_submissions.csv', index=False)

In [32]:
y_test.to_csv('titanic/wcg_submissions.csv', index=False)